In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import pandas as pd
import torch
import time
import gc
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader

# 1. Define the Architectural Competitors (Parameter Scaling Curve)
MODELS_TO_TEST = [
    "all-MiniLM-L6-v2",                        # 22M Params: Industry baseline
    "BAAI/bge-small-en-v1.5",                  # 33M Params: RAG optimized
    "sentence-transformers/all-mpnet-base-v2"  # 109M Params: Heavyweight comparison
]

# 2. Data Preparation
print("--- LOADING DATA & ENFORCING SPLIT ---")
df = pd.read_csv("mcp_routing_dataset.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

corpus_schemas = df['positive_schema'].unique().tolist()
print(f"Total rows: {len(df)} | Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Unique MCP Tools in Corpus: {len(corpus_schemas)}\n")

train_examples = [InputExample(texts=[row['anchor'], row['positive_schema']]) for _, row in train_df.iterrows()]
benchmark_results = []

# 3. The Benchmarking Loop
for model_name in MODELS_TO_TEST:
    print(f"==================================================")
    print(f"🚀 INITIATING PIPELINE FOR: {model_name}")
    print(f"==================================================")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("[1/3] Initializing and Training...")
    model = SentenceTransformer(model_name, device=device)
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
    train_loss = losses.MultipleNegativesRankingLoss(model=model)

    train_start_time = time.time()
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=4,
        warmup_steps=int(len(train_dataloader) * 0.1),
        use_amp=True,
        show_progress_bar=False,
        output_path=f"./models/benchmark_{model_name.replace('/', '_')}"
    )
    train_duration = time.time() - train_start_time
    print(f"Training completed in {train_duration:.2f} seconds.")

    print("[2/3] Embedding Corpus & Running Evaluation...")
    corpus_embeddings = model.encode(corpus_schemas, convert_to_tensor=True)

    recall_1 = 0
    recall_3 = 0
    latencies = []

    for _, row in test_df.iterrows():
        query = row['anchor']
        true_schema = row['positive_schema']

        inf_start = time.time()
        query_embedding = model.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

        top_results = torch.topk(cos_scores, k=min(3, len(corpus_schemas)))
        top_indices = top_results[1].cpu().tolist()

        latencies.append((time.time() - inf_start) * 1000)

        top_1_schema = corpus_schemas[top_indices[0]]
        top_3_schemas = [corpus_schemas[i] for i in top_indices]

        if true_schema == top_1_schema: recall_1 += 1
        if true_schema in top_3_schemas: recall_3 += 1

    r1_pct = (recall_1 / len(test_df)) * 100
    r3_pct = (recall_3 / len(test_df)) * 100
    avg_latency = sum(latencies) / len(latencies)
    print(f"Evaluation finished. Recall@1: {r1_pct:.2f}%")

    benchmark_results.append({
        "Model": model_name.split("/")[-1],
        "Params": "109M" if "mpnet" in model_name else ("33M" if "bge" in model_name else "22M"),
        "Train Time (s)": round(train_duration, 2),
        "Recall@1 (%)": round(r1_pct, 2),
        "Recall@3 (%)": round(r3_pct, 2),
        "Latency (ms)": round(avg_latency, 2)
    })

    print("[3/3] Purging VRAM for next model...\n")
    del model
    del corpus_embeddings
    gc.collect()
    torch.cuda.empty_cache()

# 4. Output Academic Thesis Table
print("==================================================")
print("🏆 FINAL BENCHMARK RESULTS")
print("==================================================")
results_df = pd.DataFrame(benchmark_results)
print(results_df.to_markdown(index=False))

--- LOADING DATA & ENFORCING SPLIT ---
Total rows: 750 | Train: 600 | Test: 150
Unique MCP Tools in Corpus: 15

🚀 INITIATING PIPELINE FOR: all-MiniLM-L6-v2
[1/3] Initializing and Training...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'train_runtime': '12.43', 'train_samples_per_second': '193.1', 'train_steps_per_second': '12.23', 'train_loss': '0.7014', 'epoch': '4'}


...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'train_runtime': '89.33', 'train_samples_per_second': '26.86', 'train_steps_per_second': '1.701', 'train_loss': '0.6405', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed in 94.15 seconds.
[2/3] Embedding Corpus & Running Evaluation...
Evaluation finished. Recall@1: 100.00%
[3/3] Purging VRAM for next model...

🏆 FINAL BENCHMARK RESULTS
| Model             | Params   |   Train Time (s) |   Recall@1 (%) |   Recall@3 (%) |   Latency (ms) |
|:------------------|:---------|-----------------:|---------------:|---------------:|---------------:|
| all-MiniLM-L6-v2  | 22M      |            13.68 |          98.67 |            100 |           8.4  |
| bge-small-en-v1.5 | 33M      |            36.98 |          99.33 |            100 |          14.73 |
| all-mpnet-base-v2 | 109M     |            94.15 |         100    |            100 |          17.66 |


In [ ]:
import shutil
from google.colab import files

print("Zipping the winning MPNet model...")
# Target the specific directory created during the benchmark loop
shutil.make_archive('best_mcp_router', 'zip', './models/benchmark_sentence-transformers_all-mpnet-base-v2')

print("Triggering download...")
files.download('best_mcp_router.zip')

Zipping the winning MPNet model...
Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>